# Lung Cancer CT Adversarial Robustness

Complete research notebook for three-class lung cancer CT classification using the IQ-OTH/NCCD dataset, FGSM and PGD adversarial attacks, defensive distillation, adversarial training, symmetric cross-entropy, and robustness evaluation.

**Data:** IQ-OTH/NCCD Lung Cancer Dataset. The dataset is not redistributed in this repository.

**Security note:** This notebook expects the user to provide their own `kaggle.json` credentials at runtime. Do not commit API keys, credentials, datasets, or model checkpoints to a public repository.


In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download the dataset
!kaggle datasets download adityamahimkar/iqothnccd-lung-cancer-dataset

# Unzip the dataset
!unzip iqothnccd-lung-cancer-dataset.zip -d lung_cancer_dataset

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Set your dataset directory
directory = 'lung_cancer_dataset/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset'
categories = ['Bengin cases', 'Malignant cases', 'Normal cases']

# Count the number of images in each class
class_counts = []
for cat in categories:
    path = os.path.join(directory, cat)
    count = len(os.listdir(path))
    class_counts.append(count)

# Plot the histogram with percentages and edge colors
plt.figure(figsize=(8,6))
bars = sns.barplot(x=categories, y=class_counts, edgecolor='black')

# Annotate with percentages
total = sum(class_counts)
for bar, count in zip(bars.patches, class_counts):
    height = bar.get_height()
    percentage = (count / total) * 100
    plt.text(bar.get_x() + bar.get_width()/2, height + 3, f'{count} ({percentage:.1f}%)',
             ha='center', va='bottom', fontsize=10)

plt.title('Class Distribution in Lung Cancer Dataset')
plt.ylabel('Number of Images')
plt.xlabel('Class')
plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

def plot_image_dimension_distribution(directory, categories):
    widths, heights, class_labels = [], [], []

    for cat in categories:
        path = os.path.join(directory, cat)
        for file in os.listdir(path):
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path, 0)
            if img is not None:
                h, w = img.shape
                heights.append(h)
                widths.append(w)
                class_labels.append(cat)

    plt.figure(figsize=(12,6))
    plt.subplot(1, 2, 1)
    sns.histplot(widths, bins=30, kde=True)
    plt.title('Distribution of Image Widths')
    plt.xlabel('Width (pixels)')

    plt.subplot(1, 2, 2)
    sns.histplot(heights, bins=30, kde=True)
    plt.title('Distribution of Image Heights')
    plt.xlabel('Height (pixels)')
    plt.tight_layout()
    plt.show()

def plot_pixel_intensity_histograms(directory, categories):
    plt.figure(figsize=(10,6))
    for cat in categories:
        path = os.path.join(directory, cat)
        all_pixels = []
        for file in os.listdir(path):
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path, 0)
            if img is not None:
                all_pixels.extend(img.flatten())
        sns.kdeplot(all_pixels, label=cat, fill=False)
    plt.title('Pixel Intensity Distribution by Class')
    plt.xlabel('Pixel Intensity')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
directory = 'lung_cancer_dataset/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset'
categories = ['Bengin cases', 'Malignant cases', 'Normal cases']

plot_image_dimension_distribution(directory, categories)
plot_pixel_intensity_histograms(directory, categories)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logs

import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Input, Lambda, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from tensorflow.keras.utils import to_categorical

# =========================
# Global Settings
# =========================
img_size = 256
batch_size = 8
epochs = 50
learning_rate = 0.001
robust_teacher_epochs = 50
robust_student_epochs = 50
temperature = 10.0
epsilon = 0.05   # Maximum perturbation
alpha = 0.005    # PGD step size
pgd_steps = 25   # PGD steps
fgsm_epsilon = epsilon  # FGSM uses same epsilon

# =========================
# Reproducibility
# =========================
random.seed(10)
np.random.seed(10)
tf.random.set_seed(10)

# =========================
# Data Loading / Preprocessing
# =========================
def load_data(directory, categories, img_size):
    data = []
    for category in categories:
        path = os.path.join(directory, category)
        class_num = categories.index(category)
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img), cv2.IMREAD_GRAYSCALE)
                resized_array = cv2.resize(img_array, (img_size, img_size))
                data.append([resized_array, class_num])
            except Exception:
                pass
    return data

def prepare_features_labels(data, img_size):
    X = []
    y = []
    for features, label in data:
        X.append(features)
        y.append(label)
    X = np.array(X).reshape(-1, img_size, img_size, 1).astype('float32') / 255.0
    y = np.array(y)
    return X, y

data_unpoisoned = load_data(directory, categories, img_size)
X_unpoisoned, y_unpoisoned = prepare_features_labels(data_unpoisoned, img_size)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_unpoisoned, y_unpoisoned, test_size=0.2, random_state=10, stratify=y_unpoisoned
)

# Balance training data using SMOTE
X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
smote = SMOTE(random_state=10)
X_train_smote, y_train_smote = smote.fit_resample(X_train_reshaped, y_train)
X_train_smote = X_train_smote.reshape(X_train_smote.shape[0], img_size, img_size, 1)

# =========================
# Model Definition
# =========================
def create_deep_model(temp=1.0):
    """Creates a CNN with temperature-scaled softmax."""
    inp = Input(shape=(img_size, img_size, 1))
    x = Conv2D(64, (3,3), activation='relu', padding='same')(inp)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.5)(x)
    x = Flatten()(x)
    x = Dense(16, activation='relu')(x)
    logits = Dense(3)(x)
    output = Lambda(lambda z: tf.nn.softmax(z / temp))(logits)
    model = Model(inputs=inp, outputs=output)
    return model

def symmetric_cross_entropy(y_true, y_pred, alpha=0.1, beta=1.0, epsilon=1e-7):
    y_pred_clipped = tf.clip_by_value(y_pred, epsilon, 1.0)
    y_true_clipped = tf.clip_by_value(y_true, epsilon, 1.0)
    ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    rce = -tf.reduce_sum(y_pred_clipped * tf.math.log(y_true_clipped), axis=1)
    return alpha * ce + beta * rce

# =========================
# Visualization Helpers
# =========================
def plot_training_curves(history, title_prefix):
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(f'{title_prefix} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{title_prefix} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

# =========================
# Adversarial Attack Functions
# =========================
def generate_pgd_examples_batched(model, X, y, epsilon=epsilon, alpha=alpha, steps=pgd_steps, batch_size_pgd=32):
    adv_examples = []
    n_samples = len(X)
    for i in range(0, n_samples, batch_size_pgd):
        X_batch = tf.convert_to_tensor(X[i:i+batch_size_pgd], dtype=tf.float32)
        y_batch = y[i:i+batch_size_pgd]
        X_adv_batch = tf.identity(X_batch)
        for _ in range(steps):
            with tf.GradientTape() as tape:
                tape.watch(X_adv_batch)
                predictions = model(X_adv_batch, training=False)
                loss = tf.keras.losses.sparse_categorical_crossentropy(
                    tf.convert_to_tensor(y_batch, dtype=tf.int64), predictions
                )
            gradient = tape.gradient(loss, X_adv_batch)
            X_adv_batch = X_adv_batch + alpha * tf.sign(gradient)
            X_adv_batch = tf.clip_by_value(X_adv_batch, X_batch - epsilon, X_batch + epsilon)
            X_adv_batch = tf.clip_by_value(X_adv_batch, 0, 1)
        adv_examples.append(X_adv_batch.numpy())
    return np.concatenate(adv_examples, axis=0)

def generate_fgsm_examples_batched(model, X, y, epsilon=fgsm_epsilon, batch_size_pgd=32):
    adv_examples = []
    n_samples = len(X)
    for i in range(0, n_samples, batch_size_pgd):
        X_batch = tf.convert_to_tensor(X[i:i+batch_size_pgd], dtype=tf.float32)
        y_batch = y[i:i+batch_size_pgd]
        with tf.GradientTape() as tape:
            tape.watch(X_batch)
            predictions = model(X_batch, training=False)
            loss = tf.keras.losses.sparse_categorical_crossentropy(
                tf.convert_to_tensor(y_batch, dtype=tf.int64), predictions
            )
        gradient = tape.gradient(loss, X_batch)
        X_adv_batch = X_batch + epsilon * tf.sign(gradient)
        X_adv_batch = tf.clip_by_value(X_adv_batch, 0, 1)
        adv_examples.append(X_adv_batch.numpy())
    return np.concatenate(adv_examples, axis=0)

# =========================
# Stage 1: Baseline Model
# =========================
baseline_model = create_deep_model(temp=1.0)
baseline_optimizer = Adam(learning_rate=learning_rate)
baseline_loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
baseline_model.compile(loss=baseline_loss_fn, optimizer=baseline_optimizer, metrics=['accuracy'])

baseline_y_train = to_categorical(y_train_smote, num_classes=3).astype(np.float32)
baseline_y_valid = to_categorical(y_valid, num_classes=3).astype(np.float32)
history_baseline = baseline_model.fit(
    X_train_smote, baseline_y_train, batch_size=batch_size, epochs=epochs,
    validation_data=(X_valid, baseline_y_valid)
)
plot_training_curves(history_baseline, 'Stage 1: Baseline Model')

baseline_preds_clean = baseline_model.predict(X_valid)
baseline_pred_labels_clean = np.argmax(baseline_preds_clean, axis=1)
baseline_acc_clean = np.mean(baseline_pred_labels_clean == y_valid)
print(f'Stage 1 - Baseline Clean Validation Accuracy: {baseline_acc_clean:.4f}')
print(classification_report(y_valid, baseline_pred_labels_clean, target_names=categories))
plot_confusion(y_valid, baseline_pred_labels_clean, 'Stage 1: Baseline Confusion Matrix (Clean)')

# Evaluate baseline model under PGD attack
X_valid_pgd = generate_pgd_examples_batched(
    baseline_model, X_valid, y_valid, epsilon=epsilon, alpha=alpha, steps=pgd_steps, batch_size_pgd=32
)
baseline_preds_pgd = baseline_model.predict(X_valid_pgd)
baseline_pred_labels_pgd = np.argmax(baseline_preds_pgd, axis=1)
baseline_acc_pgd = np.mean(baseline_pred_labels_pgd == y_valid)
print(f'Stage 1 - Baseline PGD Accuracy: {baseline_acc_pgd:.4f}')
print(classification_report(y_valid, baseline_pred_labels_pgd, target_names=categories))
plot_confusion(y_valid, baseline_pred_labels_pgd, 'Stage 1: Baseline Confusion Matrix (PGD)')

# =========================
# Stage 2: Defensive Distillation + Adversarial Training
# =========================
print('\n==== Stage 2: Training Robust Model via Adversarial Training with Defensive Distillation ====')

robust_teacher = create_deep_model(temp=temperature)
optimizer_teacher = Adam(learning_rate=learning_rate)
teacher_loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
robust_teacher.compile(loss=teacher_loss_fn, optimizer=optimizer_teacher, metrics=['accuracy'])

robust_teacher_y_train = to_categorical(y_train_smote, num_classes=3).astype(np.float32)
robust_teacher_y_valid = to_categorical(y_valid, num_classes=3).astype(np.float32)
print('Training Robust Teacher Model with Defensive Distillation...')
history_teacher = robust_teacher.fit(
    X_train_smote, robust_teacher_y_train, batch_size=batch_size,
    epochs=robust_teacher_epochs, validation_data=(X_valid, robust_teacher_y_valid)
)
plot_training_curves(history_teacher, 'Stage 2: Robust Teacher (Defensive Distillation)')

teacher_soft_labels = robust_teacher.predict(X_train_smote).astype(np.float32)

# Generate adversarial examples using both PGD and FGSM
X_adv_pgd = generate_pgd_examples_batched(
    baseline_model, X_train_smote, y_train_smote, epsilon=epsilon, alpha=alpha, steps=pgd_steps, batch_size_pgd=32
)
X_adv_fgsm = generate_fgsm_examples_batched(
    baseline_model, X_train_smote, y_train_smote, epsilon=fgsm_epsilon, batch_size_pgd=32
)
X_adv_total = np.concatenate([X_adv_pgd, X_adv_fgsm], axis=0)
y_adv_total = np.concatenate([teacher_soft_labels, teacher_soft_labels], axis=0)

# Combined clean and adversarial training set
X_combined = np.concatenate([X_train_smote, X_adv_total], axis=0)
y_combined = np.concatenate([teacher_soft_labels, y_adv_total], axis=0)

robust_student = create_deep_model(temp=1.0)
optimizer_robust_student = Adam(learning_rate=learning_rate)
robust_student.compile(loss=symmetric_cross_entropy, optimizer=optimizer_robust_student, metrics=['accuracy'])

print('Training Robust Student Model via Combined Adversarial Training and Defensive Distillation...')
history_robust = robust_student.fit(
    X_combined, y_combined, batch_size=batch_size,
    epochs=robust_student_epochs,
    validation_data=(X_valid, to_categorical(y_valid, num_classes=3).astype(np.float32))
)
plot_training_curves(history_robust, 'Stage 2: Robust Student (Combined)')

# Clean evaluation
robust_preds_clean = robust_student.predict(X_valid)
robust_pred_labels_clean = np.argmax(robust_preds_clean, axis=1)
robust_acc_clean = np.mean(robust_pred_labels_clean == y_valid)
print(f'Stage 2 - Robust Student Clean Validation Accuracy: {robust_acc_clean:.4f}')
print('Stage 2 - Robust Student Classification Report (Clean):')
print(classification_report(y_valid, robust_pred_labels_clean, target_names=categories))
plot_confusion(y_valid, robust_pred_labels_clean, 'Stage 2: Robust Student Confusion Matrix (Clean)')

# PGD evaluation
X_valid_pgd_robust = generate_pgd_examples_batched(
    robust_teacher, X_valid, y_valid, epsilon=epsilon, alpha=alpha, steps=pgd_steps, batch_size_pgd=32
)
robust_preds_pgd = robust_student.predict(X_valid_pgd_robust)
robust_pred_labels_pgd = np.argmax(robust_preds_pgd, axis=1)
robust_acc_pgd = np.mean(robust_pred_labels_pgd == y_valid)
print(f'Stage 2 - Robust Student PGD Accuracy: {robust_acc_pgd:.4f}')
print('Stage 2 - Robust Student Classification Report (PGD):')
print(classification_report(y_valid, robust_pred_labels_pgd, target_names=categories))
plot_confusion(y_valid, robust_pred_labels_pgd, 'Stage 2: Robust Student Confusion Matrix (PGD)')

# Final summary
print('\n==== Final Robustness Summary ====')
print(f'Baseline Clean Accuracy: {baseline_acc_clean:.4f}')
print(f'Baseline PGD Accuracy: {baseline_acc_pgd:.4f}')
print(f'Robust Student Clean Accuracy: {robust_acc_clean:.4f}')
print(f'Robust Student PGD Accuracy: {robust_acc_pgd:.4f}')